<h1>Scraping ASOS: Sustainability and Pricing Strategy?</h1>

|Tru Annafi |
ECO 590 |
Spring 2026|

<h3>This notebook contains the code of webscraping ASOS and Graphs.</h3>

<h4>Motivation</h4>
I'm interested in researching fashion and the environment and I'd also like to dive into marketing. My past research as been done on consumer behavior, fashion items, and the environment/sustainability. I think marketing is a good thing to focus on so
I can help brands learn how to market sustainable items and how certain sustainability marketing tactics affect consumer behavior.
This project will help see if sustainable products are typically more expensive, which can be used for marketing and how to convince consumers whatever price premium there is, is worth it. From a marketing perspective, sustainability may function as a signal of
product quality, ethical production, or brand value, potentially allowing firms to position these items at higher price points.
This project examines whether products marketed as sustainable (ASOS’s Circular Design Collection) are priced differently from
non-sustainable products.

<h4>Research Question</h4>
Do sustainability signals in fashion act as a pricing strategy?(ASOS)

<h4>Links</h4>

[ASOS Women's Tops](https://www.asos.com/women/tops/cat/?cid=4169)

[ASOS Circular Design Collection](https://www.asos.com/us/search/?q=circular+design+collection&currentpricerange=10-170&refine=floor%3A1001%2C2001)

[Circulare Design Info](https://www.asos.com/discover/circular-design/?msockid=3ba92ad05ebf620415693e225f0f6303)

No API key is required, as the data is fully accessible through ASOS’s public website.

<h2>Part 1: Web-Scraping ASOS</h2>
This code loads libraries, defines rules, makes scraping functions, cleans data, and exports the final dataset. Python runs from top to bottom when the code is run, so the order is important. First I loaded the libraries which are all the tools the different codes need. Without the tools, the code would break right away. 

In [19]:
import requests #sends requests to websites, downloads webpages, API's etc.
import pandas as pd #works with the datasets(tables/merging/cleaning, etc)
import time #to help pause/sleep my code so I don't get blocked
from bs4 import BeautifulSoup #helps read and organize html so we can get its data
import re #extracts numbers from text/helps me get the prices
import random #using this to make random agents and make the requests seem less robotic

ERROR: Error in parse(text = input): <text>:1:8: unexpected symbol
1: import requests
           ^


<h2>User Agent & Category Map</h2>
The user agent helps my requests look like it's from a human. We do this early on because future requests use this. In order for something to be used later it has to exist first. The category map tells python if a product has one of these words, assign it to that category. The category map is also used in later code so it was made early so it existed when it was time to use it.

In [ ]:
#User Agent
user_agents = ["Mozilla/5.0 (Windows NT 10.0; Win64; x64)"] #makes request look like it came from a real browser to lessen the likelihood of being blocked

#Category Keywords (looked through all the items and saw what words were being used, so it could be sorted correctly)
category_map = {
    "tops": ["top", "t-shirt", "tee", "shirt", "cami", "halter", "blouse", "tank"],
    "jeans": ["jeans", "jean", "pants", "trousers"],
    "dress": ["dress", "gown"],
    "shorts": ["shorts", "short", "jorts"]}

<h2>Scraping Function</h2>
Creating one function as it'll be used later on. We define "scrape_asos" first, so it'll act as one scraping code instead of making multiple for all the links. Without a function, we'd have to manually repeat a lot of code. This function is like a machine that is set up to already know how to webscrape ASOS. Inside the function has multiple steps, including lists that were created to store the information, making requests look human-like, looping through pages, grabbing data information, etc. All of this code is inside the same function because it's just multiple steps for one overall task. 

In [ ]:
#Scraping Data Function
def scrape_asos(base_url, pages, sustainable_flag): #the parameters help the function be more flexible. Without parameters, the function could webscrape only one website.
#base url is the site I'm web scraping, pages is how many pages I'll be looping through, sustainable is the binary letting me know if it's sustainable or not
    products = [] #made an empty list to store all of the scraped data (storage)
    session = requests.Session() #using the library (requests) imported earlier. Session keeps a stable connection, cookies get saved, login is saved, etc. This helps the webscraping look more human like, which would reduce blocking. Storing inside the variable session so it can be reused

    for page in range(1, pages + 1): #loops through pages automatically
        url = base_url + str(page) #builds urls 
        success = False #the page hasn't been successfully scraped yet
        
        for attempt in range(3): #retries the loop up to 3 times because it can fail
            try:
                headers = {"User-Agent": random.choice(user_agents)} #picks a random browser identity to also lessen the likelihood of being blocked
                response = session.get(url, headers=headers, timeout=20) #sending a request(session.get) to the website we want to visit(url), timeout waits for a max of 20 seconds (good so the code doesn't get stuck waiting for a response if the websites slow or blocks.)
                soup = BeautifulSoup(response.text, "html.parser") #takes the html from website response, html parser organizes it, it's then stored in the soup variable. organizes html into a searchable structure
                items = soup.find_all("li", class_=lambda x: x and "productTile" in x) #finds all elements that have "productTile" somewhere in the name. Lambda is used for a  quick function that gets used once and thrown away later,
                if not items: #if "productTile" isn't there (if its empty python will see it as false) from the items collected in the line above, then...
                    raise Exception("No items found") # raise forces an error, exception is an error type that lets us make a custom error message. makes the loop try again because its in a retry loop
                
                for item in items: #going product by product
                    name_tag = item.find("p", class_=lambda x: x and "productDescription" in x) #find the <p> tag(paragraph tag) with the product name
                    name = name_tag.text.strip() if name_tag else None #text strip removes extra spaces from titles, name tag helps avoid crashing if tag name is missing
                    price_tag = item.find("p", attrs={"aria-label": True}) #finds the <p> tag with the price
                    if price_tag: #did BeautifulSoup find the right element in the html?
                        raw_price = price_tag.get("aria-label") #telling it to go inside the html price_tag and get the attributes(aria label) value, which would be the products price. Aria label helps things look cleaner and can be more c
                        match = re.search(r"\d+\.?\d*", raw_price) #\d+ is for one or more digits, \. for if there's a decimal point, \d* for if there's numbers after the decimal point. match tries to find a number inside the text and stores it in match. if a number was found, turn it into a float, if not, say None
                        price = float(match.group()) if match else None #the plain price without any additional signs should be gathered
                    else: #if a number wasn't found and couldn't turn into a float, say None
                        price = None

                    link_tag = item.find("a", href=True) #"a" is an anchor tag, it creates links. Only finding "a" tags that have href in it 
                    link = link_tag["href"] if link_tag else None #link_tag has the whole html tag in it, asks to give the value inside the href attribute
                    products.append({"name": name, "price": price, "link": link, "sustainable": sustainable_flag}) # making each product become a row, all of this goes back into the empty products list made earlier(append)
                
                print(f"Finished page {page}") #lets me know the loop happened successfully
                success = True
                break #stops loop if it works, no need to retry

            #this section tells the computer to try a few times and not crash right away
            except Exception as e: #catch any error that happens. "e" holds the error message 
                print(f"Retry {attempt+1} failed (page {page}): {e}") #prints a "f"ormatted string of the attempt count going up by 1, of the page and the e message
                time.sleep(random.uniform(5, 10)) # waits randomly between 5 and 10 seconds(random.uniform creates a random decimal number), this helps the computer seem more human like
        if not success: #if the success is still false then,
            print(f"Skipping page {page}") # print the page is being skipped, its good to skip so you can still get some data instead of stopping/crashing the whole thing
        
        time.sleep(random.uniform(5, 8)) #waits between 5 and 8 seconds for each page switch, again mimics human behavior
    
    return pd.DataFrame(products) #turns the list into a table

<h2>Category Function</h2>
This function is like a labeling system. It reads the product names and assign it to the correct category. Categorizing the products is different from scraping, this is why it's in seperate functions. The category map tells python those specific words in the list are with that specific category. If the category function reads any matching words then it'll put it with the right category. The category function connects to the scraping function in a later part. 

In [ ]:
#Category Function. Sorts the items into the right category
def get_category(name): #creating a get_category function
    if not name: #checks if the name is empty/missing
        return "other" #will be categorized as other
    name = name.lower().replace("topshop", "") #"lower" turns everything lowercase so it's easier to match. The topshop brand makes the computer mistake it for a top automatically, replacing(.replace) it with the blank makes the computer focus on the correct word identification
    for category, keywords in category_map.items(): #This loops through the category_map dictionary I made. ".items()" gives access to the key and value 
        if any(re.search(rf"\b{word}\b", name) for word in keywords): #this sees if the product name has any matches from the categories. "for word in keywords" loops through all the words at once. "re.search searches the text with regex. "\b" are word boundaries. "any()" for if at least 1 word matches. 
            return category #if something matched, add it to that category
    return "other" #if nothing matched say its other

<h2>Sources</h2>
This section is a list for the scraping. Each category of clothes has a dictionary with information about that webpage. It includes the category name, the URLs, how many pages should be scraped, and whether the products should be labeled as sustainable or non-sustainable. Tops, jeans, shorts, and dresses are assigned 0. This is because I know they aren't sustainable as they come from ASOS's main page. ASOS has a circular page specifically for sustainable items, so all products from that page has been assigned 1. I created this binary variable manually

In [ ]:
#All the Webpages We're Getting Data From
sources = [
    {"name": "tops", "url": "https://www.asos.com/women/tops/cat/?cid=4169&page=", "pages": 5, "sustainable": 0}, #the pages for how many numbers to webscrape
    {"name": "jeans", "url": "https://www.asos.com/us/women/jeans/cat/?cid=3630&page=", "pages": 5, "sustainable": 0}, #sustainable is equal to 0 because I know all of the items that aren't circular aren't sustainable
    {"name": "shorts", "url": "https://www.asos.com/us/search/?q=shorts&page=", "pages": 5, "sustainable": 0},
    {"name": "dresses", "url": "https://www.asos.com/us/search/?q=dress&page=","pages": 5,"sustainable": 0},
    {"name": "circular", "url": "https://www.asos.com/us/search/?q=circular+design+collection&page=", "pages": 2, "sustainable": 1}] #less pages to webscrape for circular because there's less sustainable items on the site, sustainable is equal to 1 because I know for sure the circular collection have all sustainable items

<h2>Main Loop & Merging</h2>
This part loops through all of the ASOS sources. The empty "all_data" list stores the data from each webpage. All of the scraped data from the products gets merged into one dataset. The category variable is created and any duplicate products are removed. Then, the final dataset is saved to a CSV file. This section connects the category function scraping function through this line specifically :
df = scrape_asos(base_url=source["url"], pages=source["pages"], sustainable_flag=source["sustainable"]). This section is last because it depends on everything that was made before it. The earlier code were preperation for this main loop code to run.


In [ ]:
#Main Loop - goes through all the ASOS sections, scrapes the data, stores the info all together
all_data = [] #created an empty list for all the data for every page
for source in sources: #loop through every item in the sources list
    print(f"\nScraping {source['name']}...") #prints a progress message. "source['name']" this gets the name value in the dictionary. "\n" makes a new line, and it makes the output cleaner
    df = scrape_asos(base_url=source["url"], pages=source["pages"], sustainable_flag=source["sustainable"]) #calls the scraping function which was made earlier. running the function and giving it values from the dictionary to replace the named arguments
    all_data.append(df) #adds the dataframes to "all_data". append() adds something to a list. so this dataframe list gets longer with append

df_final = pd.concat(all_data, ignore_index=True) #Merging all the dataframes into one big dataframe. pd is for pandas, which is for tables/datasets. concat() means to join things together. "ignore_index" ignores the row number and still acknowledges all inputs even if there's the same number multiple times. It'll make a new row for it
df_final["category"] = df_final["name"].apply(get_category) #Adding the category column. ".apply(get_category)" runs getting the category on every row.
df_final = df_final.drop_duplicates(subset="name") #cleaning the data by dropping any possible duplicates. "subset="name"" tells it to use the name of the product to decide if it's a duplicate or not
df_final.to_csv("asos_sustainability_clean.csv", index=False) #saving to a csv file. "index=False" stops pandas from saving row numbers

print(df_final.head()) #helps me check if it ran correctly by seeing the first few data lines
print("Total items:", len(df_final)) #"len()" is length, will print how many items are in the df_final

<h2>Part 2: Graphs</h2>
In this section, I'm loading the libraries/tools, loading data, cleaning data, transforming data, making graph structures, plotting graphs, formatting visuals, and displaying results. We load the libraries first so the code has the right tools available to use when run. We import the data next, so the dataframe is available for use. When your data is imported, the CSV is read, converted to a dataframe and is assigned to a variable. My cleaning data section selects what columns will be used and cleans the inputs so nothing breaks or becomes incorrect. We only need the sustainable and price columns for the graph. I created the figure layout next to give my first graph a place to go. I made my third graph separately 

In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Importing data
asos = pd.read_csv(r"C:\Users\truth\asos_sustainability_clean.csv") #making a variable asos to contain the dataset. pd for pandas, which was imported earlier, works with read_csv to open the csv, read the data, turn it into a dataframe. The "r" means raw string, tells to treat everything exactly as written, so the "\U" isn't confused for a command

clean_data = asos[['sustainable', 'price']].dropna() #selects two columns in the dataset, 2 brackets for 2 columns. ".dropna()" is apart of the pandas library, tells python to remove the missing (NaN - Not a Number) data

# Calculate average price by sustainability
sustain_price = (clean_data.groupby('sustainable')['price'].mean().reset_index()) #calculates the average price for sustainable v non-sustainable items. "groupby" separates the sustainable(1) and the non-sustainable(0). Telling it to use the price column for the calculations and the ".mean()" calculates the average. ".reset_index()" fixes the format back to normal after it was grouped by groupby. This is important because it can be harder to plot points if everything's in the same index.

# Creating the graph layout
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(14, 5)) #"plt.subplots()" makes the overall figure for the graph space, "nrows=1, ncols=2" so the two graphs are side by side. 14 wide and 5 tall. graph one in ax1 and graph 2 in ax2, these labels help edit each graph seperatley  

#bargraph - ax1
ax1.bar(sustain_price['sustainable'], sustain_price['price']) #tells the first graph to be a bar chart through Matplorlib(.bar()). sustainable is the x axis (0/1) and price is the y axis
ax1.set_title('Average Price:\nSustainable vs Non-Sustainable') #".set_title()" adds a title for the specific graph, the text inside will be the title. "\n" for new line within the title
ax1.set_xlabel('Sustainable (0 = No, 1 = Yes)') #same as the title it sets the x axis for that specific graph 
ax1.set_ylabel('Average Price') #sets the y axis for this specific graph

#grouped bar graph - ax2
sns.barplot(ax=ax2, data=asos, x='category', y='price', hue='sustainable') #sns is short for seaborn, a library imported from earlier. ".barplot()" is a seaborn function and makes statistical bar graphs. telling it to use the asos data frame. x-axis is category and y-axis is price. 
ax2.set_title('Average Price by Category and Sustainability') #setting the title for this specific graph
ax2.set_xlabel('Category') #setting the x axis as category for this specific graph
ax2.set_ylabel('Average Price') #setting the y axis for this specific graph

# Makes labels easier to read
ax2.tick_params(axis='x', rotation=45) #".tick_params()" is a Matplotlib function. telling it that the ticks of the x-axis to rotate 45 degrees, the x-axis ticks are the category labels, so those words will be rotated
ax2.legend(title='Sustainable') #".legend()" tells us waht colors mean on graphs, added a title to that.
fig.suptitle('ASOS Sustainability and Pricing', fontsize=16) # Main title for both graphs and the whole figure(fig.suptitle()). "fontsize" controls the size of the text
plt.tight_layout() # Prevent overlapping. plt is apart of Matplotlib. the tight layout automatically adjusts spacing with labels, axis, legends etc


#box plot
plt.figure(figsize=(8, 5)) #creating a new figure/seperate graph. figsize is for the width and height of the figure
sns.boxplot(data=asos, x='sustainable', y='price') # using the seaborn library and boxplot makes the type of graph wanted, which is a boxplot. using the asos dataframe and assigning the x and y axis variables
plt.title('Price Distribution by Sustainability Status') #giving a title to this graph
plt.xlabel('Sustainable (0 = No, 1 = Yes)') #naming the x-axis for this graph
plt.ylabel('Price') #naming the y-axis for this graph

plt.show() # Shows the graphs/figure

ERROR: Error in parse(text = input): <text>:1:8: unexpected symbol
1: import pandas
           ^
